# Como rodar o código do NLTK para análise de sentimento

1. Instale o NLTK usando pip:
`pip install nltk
`. O pacote está disponível no arquivo `requirements.txt` para facilitar a instalação.
2. Importe o NLTK e baixe os recursos necessários (célula de código abaixo):
`python import nltk
nltk.download()
`
4. Na janela de download do NLTK, selecione todos os recursos relacionados à análise de sentimento, como "subjectivity" e "vader_lexicon". Você pode usar a barra de pesquisa para encontrar esses recursos rapidamente.

In [87]:
# import nltk
# nltk.download()

5. Importe as classes e funções necessárias para análise de sentimento:

In [88]:
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
import pandas as pd

6. Rode as demais células para preparar os dados, extrair características e treinar o classificador de Naive Bayes. O código completo está disponível nas células abaixo.

In [89]:
n_instances = 100
subj_docs = [(sent, 'subj') for sent in subjectivity.sents(categories='subj')[:n_instances]]
obj_docs = [(sent, 'obj') for sent in subjectivity.sents(categories='obj')[:n_instances]]
len(subj_docs), len(obj_docs)

(100, 100)

In [90]:
subj_docs[0]

(['smart',
  'and',
  'alert',
  ',',
  'thirteen',
  'conversations',
  'about',
  'one',
  'thing',
  'is',
  'a',
  'small',
  'gem',
  '.'],
 'subj')

In [91]:
train_subj_docs = subj_docs[:80]
test_subj_docs = subj_docs[80:100]
train_obj_docs = obj_docs[:80]
test_obj_docs = obj_docs[80:100]
train_docs = train_subj_docs + train_obj_docs
test_docs = test_subj_docs + test_obj_docs

In [92]:
sentim_analyzer = SentimentAnalyzer()
all_words_neg = sentim_analyzer.all_words([mark_negation(doc) for doc in train_docs])

In [93]:
unigram_feats = sentim_analyzer.unigram_word_feats(all_words_neg, min_freq=4)
len(unigram_feats)

83

In [94]:
sentim_analyzer.add_feat_extractor(extract_unigram_feats, unigrams=unigram_feats)

In [95]:
training_set = sentim_analyzer.apply_features(train_docs)
test_set = sentim_analyzer.apply_features(test_docs)

In [96]:
trainer = NaiveBayesClassifier.train
classifier = sentim_analyzer.train(trainer, training_set)

Training classifier


In [97]:
for key, value in sorted(classifier.most_informative_features()[:10]):
    print(f'{key}: {value}')

contains(begins): True
contains(both): True
contains(her): True
contains(him): True
contains(his): True
contains(if): True
contains(it): True
contains(its): True
contains(life): True
contains(more): True


In [98]:
results_subj = sentim_analyzer.evaluate(test_set)
for key, value in results_subj.items():
    print(f'{key}: {value}')

Evaluating NaiveBayesClassifier results...
Accuracy: 0.8
Precision [subj]: 0.8
Recall [subj]: 0.8
F-measure [subj]: 0.8
Precision [obj]: 0.8
Recall [obj]: 0.8
F-measure [obj]: 0.8


In [99]:
df = pd.read_csv('data/IMDB_Dataset.csv')

imdb_docs = [(review.split(), sentiment) for review, sentiment in zip(df['review'], df['sentiment'])]

In [100]:
n = 1000

pos_docs = [(words, label) for words, label in imdb_docs if label == 'positive'][:n]
neg_docs = [(words, label) for words, label in imdb_docs if label == 'negative'][:n]

train_docs = pos_docs[:800] + neg_docs[:800]
test_docs = pos_docs[800:] + neg_docs[800:]

In [101]:
sentim_analyzer = SentimentAnalyzer()
all_words_neg = sentim_analyzer.all_words([mark_negation(doc) for doc in train_docs])

unigram_feats = sentim_analyzer.unigram_word_feats(all_words_neg, min_freq=10)
sentim_analyzer.add_feat_extractor(extract_unigram_feats, unigrams=unigram_feats)

training_set = sentim_analyzer.apply_features(train_docs)
test_set = sentim_analyzer.apply_features(test_docs)

trainer = NaiveBayesClassifier.train
classifier = sentim_analyzer.train(trainer, training_set)

Training classifier


In [102]:
for key, value in sorted(classifier.most_informative_features()[:10]):
    print(f'{key}: {value}')

contains(awful): True
contains(bad,): True
contains(bad.): True
contains(brilliant): True
contains(excellent): True
contains(poorly): True
contains(terrific): True
contains(unique): True
contains(waste): True
contains(worst): True


In [103]:
results = sentim_analyzer.evaluate(test_set)
for key, value in results.items():
    print(f'{key}: {value}')

Evaluating NaiveBayesClassifier results...
Accuracy: 0.775
Precision [positive]: 0.7594339622641509
Recall [positive]: 0.805
F-measure [positive]: 0.7815533980582524
Precision [negative]: 0.7925531914893617
Recall [negative]: 0.745
F-measure [negative]: 0.768041237113402
